# Preprocessing: Timedelta
The mock network is intended for testing algorithms without the need of setting up an 
entire vantage6 network.

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'

## MockClient
We mock to have two organizations with three databases each (<code>rps_cohort</code>, <code>pelvis_cohort</code> and <code>rps_pelvis_cohort</code>). <br><br>
The first organization has:
* <code>rps_cohort[:10]</code> (10 pts)
* <code>pelvis_cohort[:10]</code> (10 pts)
* <code>rps_pelvis_cohort[:10]</code> (10 pts)

The second organization has:
* <code>rps_cohort[10:]</code> (304 pts)
* <code>pelvis_cohort[10:]</code> (276 pts)
* <code>non_liposarcoma_cohort[10:]</code> (590 pts)

In [3]:
# Load the dataframes from parquet files
rps_cohort = pd.read_parquet("data/rps_cohort.parquet")
pelvis_cohort = pd.read_parquet("data/pelvis_cohort.parquet")
rps_pelvis_cohort = pd.read_parquet("data/rps_pelvis_cohort.parquet")


In [4]:
# Temporary set the survival_days to the delta between the surgery date and today
rps_cohort["survival_days"] = (pd.Timestamp("2026-01-01", tz="UTC") - rps_cohort["surgery_date"]).dt.days
pelvis_cohort["survival_days"] = (pd.Timestamp("2026-01-01", tz="UTC") - pelvis_cohort["surgery_date"]).dt.days
rps_pelvis_cohort["survival_days"] = (pd.Timestamp("2026-01-01", tz="UTC") - rps_pelvis_cohort["surgery_date"]).dt.days


In [5]:
from vantage6.algorithm.mock.network import MockNetwork

network = MockNetwork(
    "v6_preprocessing",
    datasets=[
        {
            "rps": {"database": rps_cohort[:10], "db_type": "omop"},
            "pelvis": {"database": pelvis_cohort[:10], "db_type": "omop"},
            "rps_pelvis": {"database": rps_pelvis_cohort[:10], "db_type": "omop"}
        },
        {
            "rps": {"database": rps_cohort[10:], "db_type": "omop"},
            "pelvis": {"database": pelvis_cohort[10:], "db_type": "omop"},
            "rps_pelvis": {"database": rps_pelvis_cohort[10:], "db_type": "omop"}
        }
    ],
    collaboration_id=1,
)

client = network.user_client

Error importing in API mode: ImportError("dlopen(/Users/frank/Repositories/idea4rc-vantage6-algorithms/.venv/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/frank/Repositories/idea4rc-vantage6-algorithms/.venv/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


[info ] - Mock network created with 2 nodes


In [10]:
client.dataframe.preprocess(
    id_=1,
    method="timedelta",
    image="v6-preprocessing",
    arguments={
        "column": "surgery_date",
        "output_column": "new_surv"
    }
)

[info ] - Validating function action: preprocessing
[info ] - Converting algorithm output to a Parquet Table.
[info ] - Validating function action: preprocessing
[info ] - Converting algorithm output to a Parquet Table.


{'id': 1,
 'name': 'rps',
 'db_label': 'rps',
 'session_id': 1,
 'session': {'id': 1,
  'link': '/api/session/1',
  'methods': ['GET', 'PATCH', 'DELETE']},
 'tasks': {'msg': 'not implemented in the MockNetwork'},
 'last_session_task': {'msg': 'not implemented in the MockNetwork'},
 'columns': [{'name': 'patient_id',
   'dtype': dtype('float64'),
   'node_id': 1,
   'dataframe_id': 1},
  {'name': 'age', 'dtype': dtype('float64'), 'node_id': 1, 'dataframe_id': 1},
  {'name': 'sex',
   'dtype': CategoricalDtype(categories=['FEMALE', 'MALE'], ordered=False, categories_dtype=object),
   'node_id': 1,
   'dataframe_id': 1},
  {'name': 'censor', 'dtype': dtype('bool'), 'node_id': 1, 'dataframe_id': 1},
  {'name': 'status',
   'dtype': CategoricalDtype(categories=['ALIVE', 'DEAD'], ordered=False, categories_dtype=object),
   'node_id': 1,
   'dataframe_id': 1},
  {'name': 'survival_days',
   'dtype': dtype('int64'),
   'node_id': 1,
   'dataframe_id': 1},
  {'name': 'histology',
   'dtype': Ca

In [13]:
client.network.get_node(1).dataframes["rps"]

,patient_id,age,sex,censor,status,survival_days,histology,fnclcc_grade,tumor_size,surgery_date,...,completeness_of_resection_concept_id,tumor_rupture,pre_operative_chemo,post_operative_chemo,pre_operative_radio,post_operative_radio,local_recurrence,distant_metastasis,n_cancer_episodes,new_surv
0,1.0,31.0,FEMALE,True,DEAD,2027,1013 Solitary fibrous tumour,Grade 3 tumor,5.6,2020-07-03 00:00:00+00:00,...,1633801.0,True,True,False,False,True,False,False,1.0,2027
1,3.0,68.0,FEMALE,False,ALIVE,1492,1019 UPS,Grade 2 tumor,8.5,2021-12-20 00:00:00+00:00,...,1633801.0,False,False,True,False,True,False,False,1.0,1492
2,4.0,34.0,FEMALE,True,DEAD,2506,1004/1007 Liposarcoma,Grade 2 tumor,1.3,2019-03-12 00:00:00+00:00,...,1633801.0,False,False,True,False,True,False,False,1.0,2506
3,5.0,22.0,MALE,False,ALIVE,694,1019 UPS,Grade 1 tumor,9.2,2024-02-26 00:00:00+00:00,...,1634484.0,False,False,True,False,False,False,False,1.0,694
4,6.0,32.0,FEMALE,False,ALIVE,405,1004/1007 Liposarcoma,Grade 1 tumor,9.5,2024-12-11 00:00:00+00:00,...,1633801.0,False,False,False,False,False,False,False,1.0,405
5,9.0,49.0,MALE,False,ALIVE,1260,1004/1007 Liposarcoma,Grade 3 tumor,1.4,2022-08-09 00:00:00+00:00,...,1634484.0,False,False,False,True,False,False,False,1.0,1260
6,12.0,45.0,MALE,True,DEAD,2290,1019 UPS,Grade 2 tumor,5.6,2019-10-14 00:00:00+00:00,...,1634643.0,False,True,False,False,False,False,False,1.0,2290
7,14.0,34.0,MALE,True,DEAD,741,1010 Leiomyosarcoma,Grade 1 tumor,2.3,2024-01-10 00:00:00+00:00,...,1633801.0,False,True,False,False,True,False,False,1.0,741
8,18.0,25.0,MALE,True,DEAD,1174,1016 MPNST,Grade 3 tumor,4.9,2022-11-03 00:00:00+00:00,...,1634643.0,False,True,False,True,False,False,False,1.0,1174
9,20.0,64.0,MALE,False,ALIVE,1342,1019 UPS,Grade 1 tumor,2.2,2022-05-19 00:00:00+00:00,...,NaN,False,False,False,True,False,False,False,1.0,1342
